# M08 · Tests statistiques — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m08_tests/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m08_tests/03_notebook.ipynb`, créée par `python tools/start_chapter.py M08`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M08.1 à M08.9, M08.13 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M08.16 | Le test binomial exact · `binomial_test` | 🔨 | ★ | 15 |
|  | M08.17 | Le test par permutation à deux échantillons · `permutation_test` | 🔨 | ★★ | 25 |
|  | M08.18 | Les intervalles de Wilson et de Student · `proportion_ci, paired_difference_ci` | 🔨 | ★ | 20 |
|  | M08.19 | Bonferroni, l'AIC et le BIC · `bonferroni, aic_bic` | 🔨 | ★ | 10 |
|  | M08.20 | `scipy.stats.binomtest`, `permutation_test` et `ttest_rel` contre tes fonctions | 📦 | ★ | 15 |
| B | M08.21 | Prédire : sous H0, quelle part de 1 000 p-valeurs tombe sous 0,05 ? | 🔮 | ★ | 10 |
|  | M08.22 | La puissance d'un test par permutation, selon l'effet et la taille des groupes | 🔬 | ★★ | 25 |
|  | M08.23 | La loi de la statistique sous H0, et la p-valeur comme une aire | 🎨 | ★ | 15 |
| C | M08.24 | Chasse à l'erreur : une p-valeur nulle, un test unilatéral choisi après coup, Bonferroni à l'envers | 🐛 | ★ | 15 |
|  | M08.25 | Tester un test : un pytest qui vérifie l'uniformité des p-valeurs sous H0 | 🛠️ | ★ | 15 |
|  | M08.26 | Défi : retrouver les trois vraies features parmi cinquante, avec au plus un faux positif | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M08")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : test binomial exact, test par permutation exhaustif, risques et puissance, taille d'échantillon, intervalles de Wald, de Wilson et de Student, comparaisons multiples, AIC et BIC, facteur de Bayes, importance par permutation.
- Écrire les six fonctions de `mymath/testing.py` et les tester, puis comparer avec `scipy.stats`.
- Prédire la loi des p-valeurs, mesurer une puissance, tracer une loi nulle exacte, chasser des erreurs, tester un test, choisir des features parmi cinquante.

**Rappel express.** p-valeur = $P_{H_0}(\text{statistique au moins aussi extrême que l'observation})$ ; risque de type I $\alpha$ = $P_{H_0}(\text{rejeter})$, puissance $1 - \beta$ = $P_{H_1}(\text{rejeter})$ ; $n = \Big(\frac{(z_{1-\alpha} + z_{1-\beta})\,\sigma}{\delta}\Big)^2$ ; permutation : $\frac{C + 1}{B + 1}$ ; Wald $\hat{p} \pm z\sqrt{\hat{p}(1 - \hat{p})/n}$ ; Student apparié $\bar{d} \pm t_{n-1}\,s_d/\sqrt{n}$ ; Bonferroni $\min(1, m\,p_i)$ ; AIC $= 2k - 2\ln L$, BIC $= k\ln n - 2\ln L$ ; $\mathrm{BF}_{12} = \frac{p(\mathcal{D} \mid M_1)}{p(\mathcal{D} \mid M_2)}$.

## Partie 0 · Vérifier tes exercices papier (✏️ M08.1 à M08.9, 🔗 M08.13)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"7/9"` ;
- une liste `[-1.5, 2.5]` (deux bornes, une liste de valeurs), ou `True` / `False`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M08.1 — Un test binomial à la main : un algorithme de compression est-il plus rapide sur douze fichiers ?**

In [ ]:
# Ex M08.1: the values you found on paper
answer_M08_1a = ...  # a) an irreducible fraction
answer_M08_1b = ...  # b) an irreducible fraction
answer_M08_1c = ...  # c) an irreducible fraction
answer_M08_1d = ...  # d) True or False
answer_M08_1e = ...  # e) True or False
answer_M08_1f = ...  # f) an integer
answer_M08_1g = ...  # g) a number (3 decimals)

for letter, answer in zip("abcdefg", [answer_M08_1a, answer_M08_1b, answer_M08_1c, answer_M08_1d, answer_M08_1e, answer_M08_1f, answer_M08_1g]):
    wbm.check(f"M08.1{letter}", answer)

**Ex M08.2 — Un test par permutation exhaustif : deux groupes de trois mesures, vingt répartitions**

In [ ]:
# Ex M08.2: the values you found on paper
answer_M08_2a = ...  # a) an integer
answer_M08_2b = ...  # b) a number
answer_M08_2c = ...  # c) an irreducible fraction
answer_M08_2d = ...  # d) an irreducible fraction
answer_M08_2e = ...  # e) True or False
answer_M08_2f = ...  # f) an integer
answer_M08_2g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M08_2a, answer_M08_2b, answer_M08_2c, answer_M08_2d, answer_M08_2e, answer_M08_2f, answer_M08_2g]):
    wbm.check(f"M08.2{letter}", answer)

**Ex M08.3 — Erreur de type I, erreur de type II et puissance d'un test binomial**

In [ ]:
# Ex M08.3: the values you found on paper
answer_M08_3a = ...  # a) an irreducible fraction
answer_M08_3b = ...  # b) a number (3 decimals)
answer_M08_3c = ...  # c) a number (3 decimals)
answer_M08_3d = ...  # d) an irreducible fraction
answer_M08_3e = ...  # e) a number (3 decimals)
answer_M08_3f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_M08_3a, answer_M08_3b, answer_M08_3c, answer_M08_3d, answer_M08_3e, answer_M08_3f]):
    wbm.check(f"M08.3{letter}", answer)

**Ex M08.4 — Combien d'observations pour une puissance de 80 % ? (approximation normale)**

In [ ]:
# Ex M08.4: the values you found on paper
answer_M08_4a = ...  # a) a number (3 decimals)
answer_M08_4b = ...  # b) a number (3 decimals)
answer_M08_4c = ...  # c) an integer
answer_M08_4d = ...  # d) an integer
answer_M08_4e = ...  # e) an integer
answer_M08_4f = ...  # f) a number (2 decimals)

for letter, answer in zip("abcdef", [answer_M08_4a, answer_M08_4b, answer_M08_4c, answer_M08_4d, answer_M08_4e, answer_M08_4f]):
    wbm.check(f"M08.4{letter}", answer)

**Ex M08.5 — L'intervalle d'une proportion : Wald contre Wilson, avec peu de succès**

In [ ]:
# Ex M08.5: the values you found on paper
answer_M08_5a = ...  # a) a number
answer_M08_5b = ...  # b) a number (4 decimals)
answer_M08_5c = ...  # c) a list of 2 numbers (4 decimals)
answer_M08_5d = ...  # d) a list of 2 numbers (4 decimals)
answer_M08_5e = ...  # e) a number (4 decimals)
answer_M08_5f = ...  # f) a number (4 decimals)

for letter, answer in zip("abcdef", [answer_M08_5a, answer_M08_5b, answer_M08_5c, answer_M08_5d, answer_M08_5e, answer_M08_5f]):
    wbm.check(f"M08.5{letter}", answer)

**Ex M08.6 — L'intervalle de Student d'une différence appariée : un serveur avant et après une mise à jour**

In [ ]:
# Ex M08.6: the values you found on paper
answer_M08_6a = ...  # a) a number
answer_M08_6b = ...  # b) a number (4 decimals)
answer_M08_6c = ...  # c) a number (3 decimals)
answer_M08_6d = ...  # d) a number (3 decimals)
answer_M08_6e = ...  # e) a list of 2 numbers (2 decimals)
answer_M08_6f = ...  # f) a number (2 decimals)
answer_M08_6g = ...  # g) True or False
answer_M08_6h = ...  # h) an irreducible fraction

for letter, answer in zip("abcdefgh", [answer_M08_6a, answer_M08_6b, answer_M08_6c, answer_M08_6d, answer_M08_6e, answer_M08_6f, answer_M08_6g, answer_M08_6h]):
    wbm.check(f"M08.6{letter}", answer)

**Ex M08.7 — Vingt tests au seuil de 5 % : les faux positifs attendus, puis Bonferroni**

In [ ]:
# Ex M08.7: the values you found on paper
answer_M08_7a = ...  # a) a number
answer_M08_7b = ...  # b) a number (4 decimals)
answer_M08_7c = ...  # c) a number
answer_M08_7d = ...  # d) a number (4 decimals)
answer_M08_7e = ...  # e) a list of 3 numbers
answer_M08_7f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_M08_7a, answer_M08_7b, answer_M08_7c, answer_M08_7d, answer_M08_7e, answer_M08_7f]):
    wbm.check(f"M08.7{letter}", answer)

**Ex M08.8 — L'AIC et le BIC de deux modèles, à partir de leurs log-vraisemblances**

In [ ]:
# Ex M08.8: the values you found on paper
answer_M08_8a = ...  # a) a number
answer_M08_8b = ...  # b) a number
answer_M08_8c = ...  # c) a number (1 decimal)
answer_M08_8d = ...  # d) a number (1 decimal)
answer_M08_8e = ...  # e) 1 or 2
answer_M08_8f = ...  # f) 1 or 2
answer_M08_8g = ...  # g) a number (3 decimals)

for letter, answer in zip("abcdefg", [answer_M08_8a, answer_M08_8b, answer_M08_8c, answer_M08_8d, answer_M08_8e, answer_M08_8f, answer_M08_8g]):
    wbm.check(f"M08.8{letter}", answer)

**Ex M08.9 — Un facteur de Bayes entre deux taux de clics**

In [ ]:
# Ex M08.9: the values you found on paper
answer_M08_9a = ...  # a) a number (3 decimals)
answer_M08_9b = ...  # b) a number (3 decimals)
answer_M08_9c = ...  # c) a number (2 decimals)
answer_M08_9d = ...  # d) a number (2 decimals)
answer_M08_9e = ...  # e) an irreducible fraction
answer_M08_9f = ...  # f) a number (1 decimal)

for letter, answer in zip("abcdef", [answer_M08_9a, answer_M08_9b, answer_M08_9c, answer_M08_9d, answer_M08_9e, answer_M08_9f]):
    wbm.check(f"M08.9{letter}", answer)

**Ex M08.13 — Pont : l'importance d'une feature par permutation, sur un petit tableau**

In [ ]:
# Ex M08.13: the values you found on paper
answer_M08_13a = ...  # a) a number
answer_M08_13b = ...  # b) an irreducible fraction
answer_M08_13c = ...  # c) an irreducible fraction
answer_M08_13d = ...  # d) an irreducible fraction
answer_M08_13e = ...  # e) an irreducible fraction
answer_M08_13f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M08_13a, answer_M08_13b, answer_M08_13c, answer_M08_13d, answer_M08_13e, answer_M08_13f]):
    wbm.check(f"M08.13{letter}", answer)

## Partie A · Ta librairie : test binomial, test par permutation, intervalles, Bonferroni, AIC et BIC ; SciPy

Fiche §2 à §7. Les six fonctions de `mymath/testing.py`, puis les mêmes tests avec `scipy.stats`.

In [ ]:
# Tools of the notebook (parts A to C)
import itertools
import math
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M08.16 — Le test binomial exact · `binomial_test` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la p-valeur exacte d'un test binomial, unilatérale ou bilatérale, et refuser les entrées qui n'ont pas de sens.  
**Prérequis :** fiche §2 · **mymath :** `testing.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/testing.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy et `math` suffisent, avec deux quantiles permis : `statistics.NormalDist().inv_cdf` (loi normale) et `scipy.stats.t.ppf` (loi de Student). Les autres outils de `scipy.stats` (`binomtest`, `permutation_test`, `ttest_rel`…) sont interdits dans `testing.py` : les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas. Les nombres renvoyés sont des `float` de Python.

Écris `binomial_test(k, n, p=0.5, alternative="two-sided")` (fiche §2) : la p-valeur exacte du test de $H_0$ : « la probabilité de succès vaut $p$ », après $k$ succès en $n$ essais. Avec $K$ de loi $\mathcal{B}(n, p)$ :
- `"greater"` : $P(K \ge k)$ ; `"less"` : $P(K \le k)$ ;
- `"two-sided"` : la somme des $P(K = j)$ sur toutes les issues $j$ au plus aussi probables que l'observation, $P(K = j) \le P(K = k)$ (à un facteur $1 + 10^{-7}$ près, pour les arrondis), plafonnée à 1 ;
- une `ValueError` si $k$, $n$ ou $p$ sort de son domaine, ou si `alternative` est inconnue.

Pour les coefficients binomiaux, `math.comb(n, j)`.

La vérification calcule :
- a) `binomial_test(13, 16, alternative="greater")` (4 décimales) ;
- b) `binomial_test(13, 16)` (4 décimales) ;
- c) `binomial_test(1, 18, p=0.3, alternative="less")` (4 décimales) ;
- d) `binomial_test(9, 15, p=0.3)` (4 décimales) ;
- e) le nom de l'erreur levée par `binomial_test(4, 3)` ;
- f) le nom du type de `binomial_test(13, 16)`, en texte.

Dans ta copie : calcule a) à la main, en fraction. Pourquoi b) est-il exactement le double de a), et d) **pas** le double de $P(K \ge 9)$ ? Quelles issues $j$ la p-valeur d) additionne-t-elle ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M08")  # reload your saved file

In [ ]:
with checked("M08.16a"):
    wbm.check("M08.16a", mymath.testing.binomial_test(13, 16, alternative="greater"), computed=True)
with checked("M08.16b"):
    wbm.check("M08.16b", mymath.testing.binomial_test(13, 16), computed=True)
with checked("M08.16c"):
    wbm.check("M08.16c", mymath.testing.binomial_test(1, 18, p=0.3, alternative="less"), computed=True)
with checked("M08.16d"):
    wbm.check("M08.16d", mymath.testing.binomial_test(9, 15, p=0.3), computed=True)
with checked("M08.16e"):
    wbm.check("M08.16e", raised(mymath.testing.binomial_test, 4, 3), computed=True)
with checked("M08.16f"):
    wbm.check("M08.16f", type(mymath.testing.binomial_test(13, 16)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m08_testing.py", "test_binomial_test_ or test_testing_uses_")

### Ex M08.17 — Le test par permutation à deux échantillons · `permutation_test` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer un test par permutation reproductible, qui suit une procédure fixée, et voir qu'il ne renvoie jamais une p-valeur nulle.  
**Prérequis :** fiche §3 · **mymath :** `testing.py` · **Parcours :** C, E

Écris `permutation_test(x, y, n_permutations=9999, alternative="two-sided", seed=0)` (fiche §3), en suivant **exactement** la procédure de sa docstring :
- la statistique $t = \bar{x} - \bar{y}$, et les valeurs réunies, `x` d'abord, puis `y` ;
- `n_permutations` fois : `perm = rng.permutation(pooled)`, avec **un seul** générateur `rng = np.random.default_rng(seed)` créé avant la boucle, puis $t_{\text{perm}}$ = la moyenne de `perm[:len(x)]` moins celle de `perm[len(x):]` ;
- $C$, le nombre de tirages au moins aussi extrêmes que $t$, à $10^{-9}$ près pour les arrondis (`"greater"` : $t_{\text{perm}} \ge t - 10^{-9}$ ; `"less"` : $t_{\text{perm}} \le t + 10^{-9}$ ; `"two-sided"` : $|t_{\text{perm}}| \ge |t| - 10^{-9}$) ;
- la p-valeur $\frac{C + 1}{B + 1}$, avec $B$ = `n_permutations` ;
- une `ValueError` si un échantillon est vide, si `n_permutations` est plus petit que 1 ou si `alternative` est inconnue.

Suivre la procédure à la lettre (le même générateur, utilisé dans le même ordre) donne exactement les p-valeurs de la référence : c'est ce que vérifient les tests.

Un ancien outil a demandé $x = (8{,}1 ;\ 9{,}4 ;\ 7{,}7 ;\ 9{,}0)$ minutes pour quatre tâches, et un nouvel outil $y = (6{,}9 ;\ 7{,}2 ;\ 8{,}0 ;\ 6{,}5 ;\ 7{,}4)$ minutes pour cinq autres. La vérification calcule :
- a) `permutation_test(x, y, n_permutations=1999, alternative="greater", seed=3)` (4 décimales) ;
- b) la même chose avec `alternative="two-sided"` (4 décimales) ;
- c) `permutation_test(u, v, n_permutations=499, alternative="greater", seed=1)` pour deux groupes de huit mesures bien séparées, $u = (14{,}2 ;\ 15{,}0 ;\ 13{,}9 ;\ 14{,}8 ;\ 15{,}3 ;\ 14{,}6 ;\ 15{,}1 ;\ 14{,}4)$ et $v = (12{,}1 ;\ 12{,}9 ;\ 12{,}5 ;\ 13{,}0 ;\ 12{,}3 ;\ 12{,}8 ;\ 12{,}6 ;\ 12{,}2)$ (4 décimales) ;
- d) le nom de l'erreur levée par `permutation_test([], [4, 5])` ;
- e) le nom du type de la p-valeur de a), en texte.

Dans ta copie : M08.20 calcule la p-valeur exacte de a), sur les 126 répartitions ; une fois M08.20 fait, dis de combien a) s'en écarte, et si c'est compatible avec l'erreur type d'une proportion estimée sur 1 999 tirages. Pourquoi c) ne peut-il pas descendre sous $\frac{1}{500}$, et que faudrait-il pour une p-valeur plus petite ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M08")  # reload your saved file

In [ ]:
X_17, Y_17 = [8.1, 9.4, 7.7, 9.0], [6.9, 7.2, 8.0, 6.5, 7.4]
U_17 = [14.2, 15.0, 13.9, 14.8, 15.3, 14.6, 15.1, 14.4]
V_17 = [12.1, 12.9, 12.5, 13.0, 12.3, 12.8, 12.6, 12.2]
with checked("M08.17a"):
    wbm.check("M08.17a", mymath.testing.permutation_test(X_17, Y_17, n_permutations=1999, alternative="greater",
                                                         seed=3), computed=True)
with checked("M08.17b"):
    wbm.check("M08.17b", mymath.testing.permutation_test(X_17, Y_17, n_permutations=1999, alternative="two-sided",
                                                         seed=3), computed=True)
with checked("M08.17c"):
    wbm.check("M08.17c", mymath.testing.permutation_test(U_17, V_17, n_permutations=499, alternative="greater",
                                                         seed=1), computed=True)
with checked("M08.17d"):
    wbm.check("M08.17d", raised(mymath.testing.permutation_test, [], [4, 5]), computed=True)
with checked("M08.17e"):
    wbm.check("M08.17e", type(mymath.testing.permutation_test(X_17, Y_17, n_permutations=1999, alternative="greater",
                                                              seed=3)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m08_testing.py", "test_permutation_test_ or test_testing_uses_")

### Ex M08.18 — Les intervalles de Wilson et de Student · `proportion_ci, paired_difference_ci` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer les intervalles de Wald et de Wilson d'une proportion, et l'intervalle de Student d'une différence appariée.  
**Prérequis :** fiche §5 · **mymath :** `testing.py` · **Parcours :** C, E

Écris (fiche §5) :
- `proportion_ci(k, n, level=0.95, method="wilson")` : l'intervalle `(low, high)` d'une proportion après $k$ succès en $n$ essais, par la méthode de Wilson ou de Wald (`method="wald"`), avec $z$ le quantile d'ordre $\frac{1 + \text{level}}{2}$ de la loi normale ; les deux bornes sont ramenées dans $[0, 1]$ (`min` et `max`), **pour les deux méthodes** : même celles de Wilson peuvent en sortir d'un arrondi, comme 1,0000000000000002 pour $k = n$ ; une `ValueError` si $k$, $n$ ou `level` sort de son domaine, ou si `method` est inconnue ;
- `paired_difference_ci(a, b, level=0.95)` : l'intervalle de Student $\bar{d} \pm t\,\frac{s_d}{\sqrt{n}}$ de la moyenne des différences $d_i = a_i - b_i$, avec $s_d$ l'écart-type des différences ($\mathrm{ddof} = 1$) et $t$ le quantile d'ordre $\frac{1 + \text{level}}{2}$ de la loi de Student à $n - 1$ degrés de liberté (`scipy.stats.t.ppf(q, n - 1)`) ; une `ValueError` si les longueurs diffèrent, s'il y a moins de deux paires ou si `level` sort de $]0, 1[$.

Les deux fonctions renvoient un `tuple` de deux `float`.

La vérification calcule (des listes de 2 nombres, 4 décimales) :
- a) `proportion_ci(4, 30)` ;
- b) `proportion_ci(4, 30, method="wald")` ;
- c) `proportion_ci(30, 30)` ;
- d) `proportion_ci(4, 30, level=0.9)` ;
- e) `paired_difference_ci(a, b)` pour les temps de compilation (en secondes) de cinq projets avant, $a = (12{,}4 ;\ 11{,}8 ;\ 13{,}1 ;\ 12{,}9 ;\ 12{,}2)$, et après une mise à jour du compilateur, $b = (11{,}9 ;\ 11{,}6 ;\ 12{,}2 ;\ 12{,}5 ;\ 12{,}0)$ ;

puis :
- f) le nom de l'erreur levée par `paired_difference_ci([4, 5], [1, 2, 3])` ;
- g) le nom du type de `proportion_ci(4, 30)`, en texte.

Dans ta copie : pourquoi l'intervalle de Wilson c) ne contient-il pas seulement la valeur 1 ? Pourquoi d) est-il plus étroit que a) ? La mise à jour du compilateur a-t-elle un effet, d'après e) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M08")  # reload your saved file

In [ ]:
A_18, B_18 = [12.4, 11.8, 13.1, 12.9, 12.2], [11.9, 11.6, 12.2, 12.5, 12.0]
with checked("M08.18a"):
    wbm.check("M08.18a", list(mymath.testing.proportion_ci(4, 30)), computed=True)
with checked("M08.18b"):
    wbm.check("M08.18b", list(mymath.testing.proportion_ci(4, 30, method="wald")), computed=True)
with checked("M08.18c"):
    wbm.check("M08.18c", list(mymath.testing.proportion_ci(30, 30)), computed=True)
with checked("M08.18d"):
    wbm.check("M08.18d", list(mymath.testing.proportion_ci(4, 30, level=0.9)), computed=True)
with checked("M08.18e"):
    wbm.check("M08.18e", list(mymath.testing.paired_difference_ci(A_18, B_18)), computed=True)
with checked("M08.18f"):
    wbm.check("M08.18f", raised(mymath.testing.paired_difference_ci, [4, 5], [1, 2, 3]), computed=True)
with checked("M08.18g"):
    wbm.check("M08.18g", type(mymath.testing.proportion_ci(4, 30)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m08_testing.py", "test_proportion_ci_ or test_paired_difference_ci_ or test_testing_uses_")

### Ex M08.19 — Bonferroni, l'AIC et le BIC · `bonferroni, aic_bic` 🔨 ★ ⏱️ 10 min
**Objectif :** programmer les p-valeurs ajustées de Bonferroni et les deux critères d'information.  
**Prérequis :** fiche §6 et §7 · **mymath :** `testing.py` · **Parcours :** C

Écris (fiche §6 et §7) :
- `bonferroni(p_values)` : les p-valeurs ajustées $\min(1, m\,p_i)$, avec $m$ le nombre de tests, en tableau NumPy, dans le même ordre ; une `ValueError` si la liste est vide ou si une valeur sort de $[0, 1]$ ;
- `aic_bic(log_likelihood, k, n)` : le couple $(2k - 2\ln L,\ k\ln n - 2\ln L)$, avec le logarithme népérien ; une `ValueError` si $k < 1$ ou $n < 1$.

La vérification calcule :
- a) `bonferroni([0.002, 0.03, 0.2, 0.011])` (une liste, 4 décimales) ;
- b) `bonferroni([0.5, 0.7])` (une liste) ;
- c) `aic_bic(-57.3, 4, 120)` (une liste, 4 décimales) ;
- d) le nom de l'erreur levée par `bonferroni([0.05, 1.2, 0.3])` ;
- e) le nom du type de `bonferroni([0.2])`, en texte.

Dans ta copie : au risque de la famille de 5 %, quels tests de a) rejettent leur hypothèse nulle ? Pourquoi plafonner à 1 en b) ? Dans c), lequel des deux critères pénalise le plus les 4 paramètres, et à partir de quel $n$ est-ce toujours le cas ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M08")  # reload your saved file

In [ ]:
with checked("M08.19a"):
    wbm.check("M08.19a", np.asarray(mymath.testing.bonferroni([0.002, 0.03, 0.2, 0.011])).tolist(), computed=True)
with checked("M08.19b"):
    wbm.check("M08.19b", np.asarray(mymath.testing.bonferroni([0.5, 0.7])).tolist(), computed=True)
with checked("M08.19c"):
    wbm.check("M08.19c", list(mymath.testing.aic_bic(-57.3, 4, 120)), computed=True)
with checked("M08.19d"):
    wbm.check("M08.19d", raised(mymath.testing.bonferroni, [0.05, 1.2, 0.3]), computed=True)
with checked("M08.19e"):
    wbm.check("M08.19e", type(mymath.testing.bonferroni([0.2])).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m08_testing.py", "test_bonferroni_ or test_aic_bic_ or test_testing_uses_")

### Ex M08.20 — `scipy.stats.binomtest`, `permutation_test` et `ttest_rel` contre tes fonctions 📦 ★ ⏱️ 15 min
**Objectif :** faire les mêmes tests avec `scipy.stats`, lire ce que renvoie chaque fonction, et comparer avec ta librairie.  
**Prérequis :** fiche §2, §3 et §5 ; Ex M08.16 à M08.18 · **Parcours :** C

En pratique, on ne réécrit pas ces tests : `scipy.stats` (importé sous le nom `stats`) les fournit.
- `stats.binomtest(k, n, p=0.5, alternative="two-sided")` renvoie un résultat dont `.pvalue` est la p-valeur, et `.proportion_ci(confidence_level=0.95, method="wilson")` l'intervalle d'une proportion (attributs `.low` et `.high`) ;
- `stats.permutation_test((x, y), statistic, permutation_type="independent", alternative="greater", n_resamples=np.inf)` fait le test par permutation ; avec `n_resamples=np.inf`, il parcourt **toutes** les répartitions, et sa p-valeur est exacte ; la fonction `statistic(x, y, axis)` doit accepter un argument `axis` (par exemple `np.mean(x, axis=axis) - np.mean(y, axis=axis)`) ;
- `stats.ttest_rel(a, b)` fait le test de Student apparié : `.pvalue` est sa p-valeur bilatérale, et `.confidence_interval(0.95)` l'intervalle de M08.18.

Calcule avec ces fonctions :
- a) la p-valeur de `stats.binomtest(7, 30, p=0.1)` (4 décimales) ;
- b) l'intervalle de Wilson à 95 % de ce même résultat, `[low, high]` (4 décimales) ;
- c) la p-valeur exacte unilatérale du test par permutation de M08.17 a), pour les durées $x$ et $y$ (4 décimales) ;
- d) la p-valeur du test de Student apparié pour les temps de compilation $a$ et $b$ de M08.18 e) (4 décimales) ;
- e) vrai ou faux : pour chaque $k$ de 0 à 30, ta fonction `binomial_test(k, 30, 0.1)` donne la même p-valeur que `stats.binomtest(k, 30, 0.1)`, à $10^{-9}$ près (calcule-le avec une boucle) ;
- f) vrai ou faux : `stats.ttest_rel(a, b).confidence_interval(0.95)` et ta fonction `paired_difference_ci(a, b)` donnent le même intervalle, à $10^{-9}$ près.

Dans ta copie : c) est-il loin de la p-valeur de M08.17 a), estimée avec 1 999 permutations ? Que dit la p-valeur d), et est-elle cohérente avec l'intervalle de M08.18 e) (0 est-il dedans) ?

In [ ]:
answer_M08_20a = ...  # a) a number (4 decimals)
answer_M08_20b = ...  # b) a list [low, high] (4 decimals)
answer_M08_20c = ...  # c) a number (4 decimals)
answer_M08_20d = ...  # d) a number (4 decimals)
answer_M08_20e = ...  # e) True or False, computed with a loop
answer_M08_20f = ...  # f) True or False

In [ ]:
for letter, answer in zip("abcdef", [answer_M08_20a, answer_M08_20b, answer_M08_20c, answer_M08_20d,
                                     answer_M08_20e, answer_M08_20f]):
    wbm.check(f"M08.20{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §1, §3 et §4. La loi des p-valeurs sous $H_0$ et sous $H_1$, la puissance d'un test par permutation, la loi exacte d'une statistique.

### Ex M08.21 — Prédire : sous H0, quelle part de 1 000 p-valeurs tombe sous 0,05 ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la loi des p-valeurs sous $H_0$ et sous $H_1$, pour un test continu et un test discret, puis la mesurer.  
**Prérequis :** fiche §1 et §4 · **Parcours :** C, E

On répète 1 000 fois une expérience, et l'on calcule chaque fois la p-valeur d'un test. Choisis, pour chaque question :
- a) test $z$ bilatéral sur 20 mesures de loi $\mathcal{N}(\mu, 1)$, de $H_0 : \mu = 0$, quand $H_0$ est **vraie** : la part des p-valeurs au plus égales à 0,05 est (1) presque nulle ; (2) environ 0,05 ; (3) environ 0,5 ; (4) impossible à prévoir sans connaître les données ;
- b) test binomial exact bilatéral de $H_0 : p = \frac{1}{2}$ sur 10 essais, quand $H_0$ est **vraie** : la part des p-valeurs au plus égales à 0,05 est (1) environ 0,05 ; (2) nettement plus petite que 0,05 ; (3) nettement plus grande que 0,05 ;
- c) le test de a) quand $H_1$ est vraie, avec $\mu = 0{,}5$ : la part des p-valeurs au plus égales à 0,05 est (1) environ 0,05 ; (2) plus petite que 0,05 ; (3) nettement plus grande que 0,05 ;
- d) l'histogramme des 1 000 p-valeurs de a) est (1) à peu près plat ; (2) concentré près de 0 ; (3) en cloche autour de 0,5.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M08_21a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M08_21b = ...  # b) 1, 2 or 3
prediction_M08_21c = ...  # c) 1, 2 or 3
prediction_M08_21d = ...  # d) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcd", [prediction_M08_21a, prediction_M08_21b, prediction_M08_21c,
                                      prediction_M08_21d]):
    wbm.check(f"M08.21{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M08_21a, prediction_M08_21b, prediction_M08_21c, prediction_M08_21d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    rng_21 = np.random.default_rng(821)
    n_21 = 20
    z_h0 = rng_21.normal(0, 1, size=(1000, n_21)).mean(axis=1) * math.sqrt(n_21)        # H0 true: mean 0, σ = 1
    p_h0 = 2 * stats.norm.sf(np.abs(z_h0))
    z_h1 = rng_21.normal(0.5, 1, size=(1000, n_21)).mean(axis=1) * math.sqrt(n_21)      # H1 true: mean 0.5
    p_h1 = 2 * stats.norm.sf(np.abs(z_h1))
    p_by_k = np.array([stats.binomtest(k, 10).pvalue for k in range(11)])                # the 11 possible p-values
    p_binomial = p_by_k[rng_21.binomial(10, 0.5, size=1000)]                            # H0 true: p = 1/2
    exact_b = float(np.sum(stats.binom.pmf(np.flatnonzero(p_by_k <= 0.05), 10, 0.5)))
    print(f"a) z-test, H0 true: share of p-values ≤ 0.05 = {np.mean(p_h0 <= 0.05):.3f}")
    print(f"b) binomial test (n = 10), H0 true: share = {np.mean(p_binomial <= 0.05):.3f} (exact: {exact_b:.4f})")
    print(f"c) z-test, H1 true (effect 0.5): share = {np.mean(p_h1 <= 0.05):.3f}")
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
    axes[0].hist(p_h0, bins=20, range=(0, 1), color="C0")
    axes[0].set(title="H0 vraie", xlabel="p-valeur", ylabel="nombre de tests")
    axes[1].hist(p_h1, bins=20, range=(0, 1), color="C3")
    axes[1].set(title="H1 vraie (effet 0,5)", xlabel="p-valeur")
    plt.show()

**Après l'expérience**, dans ta copie : relie a) et d) à la démonstration de M08.10. Pourquoi le test binomial sur 10 essais rejette-t-il moins souvent que 5 % sous $H_0$ (liste les issues qui font rejeter) ? Que mesure la part de c), et comment la faire grandir ?

### Ex M08.22 — La puissance d'un test par permutation, selon l'effet et la taille des groupes 🔬 ★★ ⏱️ 25 min
**Objectif :** mesurer par simulation la puissance d'un test par permutation, et la comparer à celle d'un test $z$.  
**Prérequis :** fiche §3 et §4 ; Ex M08.17 · **Parcours :** C

La **puissance** d'un test est la probabilité qu'il rejette $H_0$ quand l'effet existe. Pour un test par permutation, elle n'a pas de formule : on la **mesure** par simulation. Écris `power(effect, n, n_sims=100, n_permutations=199, seed=0)`, qui suit exactement cette procédure :
1. `rng = np.random.default_rng(seed)`, un seul générateur ;
2. pour `sim` de 0 à `n_sims − 1` : `x = rng.normal(effect, 1, n)` puis `y = rng.normal(0, 1, n)` (dans cet ordre), puis `p = mymath.testing.permutation_test(x, y, n_permutations, alternative="greater", seed=sim)` ;
3. renvoie la proportion des `n_sims` p-valeurs au plus égales à 0,05, en `float`.

La vérification calcule (avec ta fonction `permutation_test` : écris d'abord M08.17) :
- a) `power(0, 10)` (2 décimales) ;
- b) `power(1.0, 10)` (2 décimales) ;
- c) `power(0.5, 20)` (2 décimales) ;
- d) parmi $n = 5$, 10 et 20, le plus petit nombre de mesures par groupe pour lequel `power(1.0, n)` atteint 0,8.

Une dernière cellule affiche la grille complète (effets 0 ; 0,5 ; 1 et $n$ = 5, 10, 20), à côté de la puissance du test $z$ unilatéral qui connaît $\sigma = 1$, $\Phi\big(\delta\sqrt{n/2} - z_{0{,}95}\big)$ : la différence de deux moyennes de $n$ mesures a l'écart-type $\sqrt{2/n}$.

Dans ta copie : que mesure a) ? Pourquoi ne vaut-il pas exactement 0,05 (pense à l'erreur type d'une proportion estimée sur 100 simulations) ? Comment la puissance évolue-t-elle avec l'effet, et avec $n$ ? Le test par permutation, qui ne suppose rien sur la loi des mesures, fait-il beaucoup moins bien que le test $z$ ?

In [ ]:
def power(effect, n, n_sims=100, n_permutations=199, seed=0):
    """The share of n_sims simulated permutation tests (alternative="greater") whose p-value is at most 0.05."""
    raise NotImplementedError("un générateur, une boucle de n_sims simulations, puis la proportion des p ≤ 0,05")

In [ ]:
with checked("M08.22a"):
    wbm.check("M08.22a", power(0, 10), computed=True)
with checked("M08.22b"):
    wbm.check("M08.22b", power(1.0, 10), computed=True)
with checked("M08.22c"):
    wbm.check("M08.22c", power(0.5, 20), computed=True)
with checked("M08.22d"):
    wbm.check("M08.22d", next((n for n in (5, 10, 20) if power(1.0, n) >= 0.8), None), computed=True)

**La grille complète** (cette cellule utilise ta fonction).

In [ ]:
with checked("M08.22 grille"):
    grid_22 = {effect: [power(effect, n) for n in (5, 10, 20)] for effect in (0, 0.5, 1.0)}
    print("effect · measured power for n = 5, 10, 20 · power of the z-test, Φ(δ√(n/2) − 1.645)")
    for effect, measured in grid_22.items():
        theory = [NormalDist().cdf(effect * math.sqrt(n / 2) - NormalDist().inv_cdf(0.95)) for n in (5, 10, 20)]
        print(f"{effect:3.1f} · " + "  ".join(f"{m:.2f}" for m in measured) + " · "
              + "  ".join(f"{t:.2f}" for t in theory))

### Ex M08.23 — La loi de la statistique sous H0, et la p-valeur comme une aire 🎨 ★ ⏱️ 15 min
**Objectif :** tracer la loi exacte d'une statistique de permutation sous $H_0$, et y lire la p-valeur comme la part des répartitions au moins aussi extrêmes.  
**Prérequis :** fiche §1 et §3 ; R0 (matplotlib) ; R8 (combinaisons) · **Parcours :** C

Écris `draw_null_distribution(ax, x, y)`, qui, pour deux petits échantillons :
1. calcule l'écart observé $t = \bar{x} - \bar{y}$ et sa loi **exacte** sous $H_0$ : les valeurs de $t_{\text{perm}}$ pour **toutes** les répartitions des valeurs réunies (`x` puis `y`) en un groupe de `len(x)` valeurs et un groupe de `len(y)` valeurs (`itertools.combinations(range(len(pooled)), len(x))` donne les indices du premier groupe) ;
2. trace dans `ax` l'histogramme de ces valeurs, avec, d'une autre couleur, les répartitions au moins aussi extrêmes que l'observation ($t_{\text{perm}} \ge t - 10^{-9}$), et une ligne verticale à $t$ (`ax.axvline`) ;
3. renvoie la p-valeur exacte unilatérale : la part de ces répartitions au moins aussi extrêmes, en `float`.

La cellule de la figure s'en sert pour les durées (en minutes) d'un même trajet, fait cinq jours par un itinéraire, $x = (31, 27, 34, 29, 33)$, et cinq autres jours par un autre, $y = (26, 28, 25, 30, 24)$. Une cellule de vérification essaie ta fonction sur un autre dataset, dans un axe neuf (la figure elle-même se juge à l'œil).

Dans ta copie : combien y a-t-il de répartitions ? Que vaut la p-valeur, et à quelle partie de la figure correspond-elle ? Pourquoi la loi est-elle symétrique autour de 0 quand les deux groupes ont la même taille ?

In [ ]:
def draw_null_distribution(ax, x, y):
    """In ax, the histogram of mean(group 1) − mean(group 2) over every split of the pooled values, the splits at least
    as extreme as the observed difference in another color, and a vertical line at it; return the exact one-sided
    p-value (alternative "greater")."""
    raise NotImplementedError("itertools.combinations pour les répartitions, puis deux ax.hist et ax.axvline")

In [ ]:
with checked("M08.23 figure"):
    fig, ax = plt.subplots(figsize=(7, 3.6))
    try:
        result_23 = []
        created = drawn(lambda: result_23.append(draw_null_distribution(ax, [31, 27, 34, 29, 33], [26, 28, 25, 30, 24])))
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M08.23 : draw_null_distribution ouvre sa propre figure : dessine dans l'axe reçu.")
    else:
        ax.set(xlabel="écart des moyennes (minutes)", ylabel="nombre de répartitions",
               title=f"Loi exacte sous H0 : p-valeur = {result_23[0]}")
        plt.show()

**Vérifications automatiques** (sur un autre dataset, dans un axe neuf).

In [ ]:
with checked("M08.23 vérifications"):
    import matplotlib.colors as mcolors
    problems = []
    X2_23, Y2_23 = [4.2, 5.1, 3.8, 4.9], [3.9, 4.4, 3.1, 4.0, 3.6, 3.3]
    pooled_t = X2_23 + Y2_23
    observed_t = np.mean(X2_23) - np.mean(Y2_23)
    splits_t = [np.mean([pooled_t[i] for i in idx]) - np.mean([pooled_t[i] for i in range(10) if i not in idx])
                for idx in itertools.combinations(range(10), 4)]
    exact_t = sum(t >= observed_t - 1e-9 for t in splits_t) / len(splits_t)
    fig_t, ax_t = plt.subplots()
    returned_t = []
    try:
        created = drawn(lambda: returned_t.append(draw_null_distribution(ax_t, X2_23, Y2_23)))
    except BaseException:
        plt.close(fig_t)
        raise
    lines_t = [line for line in ax_t.get_lines()
               if len(set(np.round(np.asarray(line.get_xdata(), dtype=float), 9))) == 1]
    colors_t = {mcolors.to_hex(patch.get_facecolor()) for patch in ax_t.patches
                if getattr(patch, "get_height", lambda: 0)() > 0}
    plt.close(fig_t)
    if created:
        problems.append("draw_null_distribution ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans "
                        "l'axe reçu")
    elif not returned_t or returned_t[0] is None:
        problems.append("draw_null_distribution ne renvoie rien : il manque return (la p-valeur exacte)")
    elif not isinstance(returned_t[0], (int, float)) or abs(float(returned_t[0]) - exact_t) > 1e-9:
        problems.append(f"la p-valeur renvoyée, {returned_t[0]}, n'est pas la part des {len(splits_t)} "
                        "répartitions au moins aussi extrêmes que l'observation (t_perm ≥ t − 1e-9)")
    elif not any(abs(float(np.asarray(line.get_xdata(), dtype=float)[0]) - observed_t) < 1e-6 for line in lines_t):
        problems.append("aucune ligne verticale à l'écart observé : ax.axvline(t)")
    elif len(colors_t) < 2:
        problems.append("les répartitions au moins aussi extrêmes doivent avoir une autre couleur que les autres")
    if problems:
        print("❌ Ex M08.23 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M08.23 : la loi exacte, la ligne de l'écart observé et la p-valeur sont justes, sur un autre "
              "dataset.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1 à §6. Trois erreurs à trouver, le test automatique d'un test, puis une sélection de features parmi cinquante.

### Ex M08.24 — Chasse à l'erreur : une p-valeur nulle, un test unilatéral choisi après coup, Bonferroni à l'envers 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques sur les p-valeurs, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §2, §3, §5 et §6 · **Parcours :** C

Un collègue a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle doit calculer. Trois lignes ne calculent pas ce que dit leur commentaire. **Sans exécuter de code**, trouve-les.

```python
B, C = 999, 0                                       # random permutations, and draws at least as extreme as observed
p_perm = C / B                                      # (A) the p-value of this permutation test
p_tail = sum(math.comb(18, j) for j in range(14, 19)) / 2 ** 18    # (B) P(K ≥ 14) for K of law B(18, 1/2)
p_values = [0.004, 0.02, 0.3]                       # the p-values of a family of three tests
adjusted = min(p_values) / len(p_values)            # (C) the Bonferroni-adjusted p-value of the smallest one
z_975 = NormalDist().inv_cdf(0.975)                 # (D) the quantile of a two-sided 95 % interval
p_two = 1 - NormalDist().cdf(abs(-1.8))             # (E) the two-sided p-value of z = −1.8 (H1 two-sided, set in advance)
half = z_975 * math.sqrt(0.3 * 0.7 / 50)            # (F) the half-width of Wald's 95 % interval, p̂ = 0.3, n = 50
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 3 décimales).

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? Pourquoi l'erreur (E) revient-elle à choisir un test unilatéral **après** avoir vu le signe de $z$, et que vaut alors le vrai risque de type I d'une règle « rejeter si cette valeur est sous 0,05 » ?

In [ ]:
answer_M08_24a = ...  # a) letters, as text
answer_M08_24b = ...  # b) a list of 6 values (3 decimals)

In [ ]:
wbm.check("M08.24a", answer_M08_24a)
wbm.check("M08.24b", answer_M08_24b)

### Ex M08.25 — Tester un test : un pytest qui vérifie l'uniformité des p-valeurs sous H0 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire le test automatique d'un test statistique : sous $H_0$, la part des p-valeurs sous le seuil doit être proche du seuil, à une tolérance comptée en erreurs types.  
**Prérequis :** fiche §1 et §4 ; Ex M08.21 ; M02c (Ex M02c.27) · **Parcours :** C

Comment vérifier qu'un test est **bien calibré**, c'est-à-dire que, sous $H_0$, il rejette au seuil de 5 % dans 5 % des cas ? Par simulation (M08.10 et M08.21) : on tire beaucoup d'échantillons sous $H_0$, on calcule leurs p-valeurs, et la part des p-valeurs au plus égales à 0,05 doit être proche de 0,05. Proche à combien près ? Comme en M02c.27 : à $k$ erreurs types près, avec l'erreur type d'une proportion, $\sqrt{\frac{\alpha(1 - \alpha)}{N}}$ pour $N$ simulations.

La cellule fournie définit `z_test_p_values(samples)`, les p-valeurs bilatérales du test $z$ de $H_0$ : « la moyenne vaut 0 » ($\sigma = 1$ connu), une par ligne de `samples`, et deux copies fausses. Écris :
- `K_SE` : le nombre d'erreurs types que ton test tolère. Un vrai projet lance ses tests des milliers de fois : la bonne fonction ne doit presque jamais échouer ; mais une copie qui rejette deux fois trop souvent, ou presque jamais, s'écarte de 0,05 d'une quinzaine d'erreurs types, et doit être attrapée à coup sûr ;
- `assert_calibrated(p_value_function, seed, n_sims=4000, n=20, alpha=0.05)` : tire `samples = np.random.default_rng(seed).normal(0, 1, size=(n_sims, n))` ($H_0$ vraie), calcule `p = p_value_function(samples)`, et **lève une `AssertionError`** (`assert condition, message`) si la part des p-valeurs au plus égales à `alpha` s'écarte de `alpha` de plus de `K_SE` erreurs types ; le message dit ce qui était attendu, puis ce qui a été obtenu ;
- `test_z_test_is_calibrated()` : le test au format pytest, sans argument, qui appelle `assert_calibrated` sur `z_test_p_values` avec une graine fixe. Dans un fichier `tests/test_….py`, pytest trouverait cette fonction et la lancerait tout seul.

La vérification lance `assert_calibrated` sur la bonne fonction et sur les deux copies fausses, pour 30 graines : la bonne doit passer **toutes** les fois, les fausses doivent échouer **toutes** les fois.

Dans ta copie : quel `K_SE` as-tu choisi, et pourquoi ? Quelle erreur chacune des deux copies fait-elle, et dans quel sens se trompe-t-elle ? Une troisième copie, qui renverrait `1 - p`, passerait ton test : pourquoi (que vaut la loi de $1 - P$ quand $P$ est uniforme sur $[0, 1]$) ? Que faudrait-il vérifier en plus pour l'attraper ?

In [ ]:
def z_test_p_values(samples):
    """Two-sided p-values of the z-test of H0: mean = 0 (σ = 1 known), one per row of samples."""
    samples = np.asarray(samples, dtype=float)
    z = samples.mean(axis=1) * math.sqrt(samples.shape[1])
    return 2 * stats.norm.sf(np.abs(z))


def z_test_copy_1(samples):            # a copy with a bug
    samples = np.asarray(samples, dtype=float)
    z = samples.mean(axis=1) * math.sqrt(samples.shape[1])
    return stats.norm.sf(np.abs(z))


def z_test_copy_2(samples):            # a copy with another bug
    samples = np.asarray(samples, dtype=float)
    z = samples.mean(axis=1) / math.sqrt(samples.shape[1])
    return 2 * stats.norm.sf(np.abs(z))

In [ ]:
K_SE = ...   # your choice: the number of standard errors your test tolerates


def assert_calibrated(p_value_function, seed, n_sims=4000, n=20, alpha=0.05):
    """Raise an AssertionError unless, under H0, the share of p-values <= alpha is within K_SE standard errors of alpha."""
    raise NotImplementedError("tire les échantillons sous H0, calcule la part des p ≤ alpha, puis assert ..., message")


def test_z_test_is_calibrated():
    """The pytest test: assert_calibrated on z_test_p_values, with a fixed seed."""
    raise NotImplementedError("un appel à assert_calibrated, avec une graine fixe")

In [ ]:
with checked("M08.25 vérifications"):
    problems = []
    if K_SE is ...:
        raise NotImplementedError("choisis K_SE, le nombre d'erreurs types que ton test tolère")
    if isinstance(K_SE, bool) or not isinstance(K_SE, (int, float)):
        raise TypeError(f"K_SE doit être un nombre, pas {type(K_SE).__name__}")
    seen = []

    def spy(samples):                                   # records the first value of the samples it receives
        seen.append(float(np.asarray(samples)[0, 0]))
        return z_test_p_values(samples)

    def outcome(function, seed):
        """None if assert_calibrated passes, else the message of its AssertionError."""
        try:
            assert_calibrated(function, seed)
        except AssertionError as exc:
            return str(exc)
        return None

    outcome(spy, 7)
    if not seen or seen[0] != float(np.random.default_rng(7).normal(0, 1, size=(4000, 20))[0, 0]):
        problems.append("assert_calibrated doit tirer samples = np.random.default_rng(seed).normal(0, 1, size=(n_sims, "
                        "n)) avec la graine reçue : un test doit être reproductible")
    else:
        passed = sum(outcome(z_test_p_values, seed) is None for seed in range(30))
        messages_1 = [outcome(z_test_copy_1, seed) for seed in range(30)]
        messages_2 = [outcome(z_test_copy_2, seed) for seed in range(30)]
        caught_1 = sum(message is not None for message in messages_1)
        caught_2 = sum(message is not None for message in messages_2)
        print(f"K_SE = {K_SE} : la bonne fonction passe ton test {passed} fois sur 30 ; la copie 1 est attrapée "
              f"{caught_1} fois sur 30, la copie 2 {caught_2} fois sur 30")
        if passed < 30:
            problems.append("ton test échoue sur la bonne fonction pour certaines graines : la tolérance est trop "
                            "étroite (un test qui échoue sans bug est « instable », flaky)")
        if caught_1 < 30:
            problems.append("ton test laisse passer la copie 1 : la tolérance est trop large")
        if caught_2 < 30:
            problems.append("ton test laisse passer la copie 2 : un test peut aussi se tromper en rejetant trop "
                            "rarement ; regarde l'écart à alpha dans les deux sens")
        if not problems and any(not message.strip() for message in messages_1 + messages_2):
            problems.append("ajoute un message à ton assert (assert condition, message) : il dit ce qui était attendu "
                            "et ce qui a été obtenu")
        if not problems and K_SE < 3.5:
            problems.append("ces 30 graines passent, mais ce K_SE est trop petit : sur des milliers de lancements, la "
                            "bonne fonction échouerait de temps en temps (avec la probabilité 2(1 − Φ(K_SE)))")
        if not problems and K_SE > 8:
            problems.append("ces 30 graines passent, mais ce K_SE est trop grand : sur des milliers de lancements, une "
                            "copie qui rejette deux fois trop souvent passerait de temps en temps")
        if not problems:
            try:
                test_z_test_is_calibrated()
            except AssertionError as exc:
                problems.append(f"test_z_test_is_calibrated échoue sur la bonne fonction : {exc}")
    if problems:
        print("❌ Ex M08.25 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M08.25 : ton test accepte la bonne fonction et attrape les deux copies fausses, pour 30 graines.")

### Ex M08.26 — Défi : retrouver les trois vraies features parmi cinquante, avec au plus un faux positif 🏆 ★★★ ⏱️ 25 min
**Objectif :** choisir une règle de décision pour 50 tests à la fois, qui trouve les vraies features sans se laisser piéger par le bruit.  
**Prérequis :** fiche §6 ; Ex M08.7, M08.11 et M08.19 · **Parcours :** C

Un dataset a 200 exemples et 50 features ; quelques-unes seulement sont liées à la cible $y$, les autres sont du bruit. Écris `select_features(X, y)`, qui renvoie la liste des indices des features que tu déclares liées à la cible.

**L'objectif** : sur `X_26, y_26`, qui a trois vraies features, et sur quatre autres datasets du même type, dont le nombre de vraies features (de 1 à 4) n'est pas donné à ta fonction, retrouver **toutes** les vraies features, avec **au plus un** faux positif par dataset, en moins de 10 secondes par dataset.

Une piste : un test par feature (la p-valeur de la corrélation de $x_j$ avec $y$, `stats.pearsonr(X[:, j], y).pvalue`), puis une règle de décision. Laquelle ? Ce chapitre a tout ce qu'il faut.

La vérification essaie ta fonction sur les cinq datasets ; une dernière cellule enregistre si l'objectif est atteint sur les cinq.

Dans ta copie : combien de features la règle « $p \le 0{,}05$ » déclare-t-elle liées sur `X_26`, et combien de faux positifs attendais-tu (M08.7) ? Pourquoi « garder les trois plus fortes corrélations » n'est-il pas une bonne règle ? Explique ta règle : que garantit-elle, et quel risque prend-elle ?

In [ ]:
def make_dataset(seed, n_true, n=200, p=50):
    """X (n × p), y and the sorted indices of the n_true features related to y (the other features are noise)."""
    rng = np.random.default_rng(seed)
    X = np.round(rng.normal(size=(n, p)), 3)
    true = np.sort(rng.choice(p, size=n_true, replace=False))
    coef = rng.uniform(0.95, 1.2, size=n_true) * rng.choice([-1, 1], size=n_true)
    y = X[:, true] @ coef + rng.normal(0, math.sqrt(6.25 - np.sum(coef ** 2)), size=n)
    return X, np.round(y, 3), true


X_26, y_26, _ = make_dataset(826, 3)          # three true features, whose indices stay hidden

In [ ]:
def select_features(X, y):
    """The list of the indices of the features of X that you declare related to y."""
    raise NotImplementedError("une p-valeur par feature, puis une règle de décision")

In [ ]:
with checked("M08.26 défi"):
    import time
    problems, results_26, ok_26, slowest = [], [], False, 0.0
    for seed_t, n_true_t in [(826, 3), (21, 2), (22, 4), (23, 1), (24, 3)]:
        X_t, y_t, true_t = make_dataset(seed_t, n_true_t)
        start_t = time.perf_counter()
        try:
            chosen_t = select_features(X_t.copy(), y_t.copy())
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"select_features lève {type(exc).__name__} ({exc})")
            break
        slowest = max(slowest, time.perf_counter() - start_t)
        if chosen_t is None:
            problems.append("select_features ne renvoie rien : il manque return")
            break
        chosen_t = {int(j) for j in np.asarray(chosen_t, dtype=int).ravel()}
        results_26.append((len(set(true_t.tolist()) - chosen_t), len(chosen_t - set(true_t.tolist()))))
    if not problems:
        print("par dataset (vraies features manquées, faux positifs) :", results_26)
        missing_26 = any(missed > 0 for missed, _ in results_26)
        extra_26 = any(false > 1 for _, false in results_26)
        if missing_26 and extra_26:
            problems.append("ta règle manque des vraies features sur certains datasets et garde plus d'un faux positif "
                            "sur d'autres : elle ne s'adapte pas au nombre de vraies features")
        elif missing_26:
            problems.append("des vraies features manquent : ta règle est trop sévère")
        elif extra_26:
            problems.append("plus d'un faux positif sur un dataset : ta règle est trop permissive")
        if slowest > 10:
            problems.append(f"objectif atteint, mais en {slowest:.1f} s pour un dataset (objectif : 10 s)")
    if problems:
        print("❌ Ex M08.26 : " + " ; ".join(problems) + ".")
    else:
        ok_26 = True
        print("✅ Ex M08.26 : défi réussi : toutes les vraies features, et au plus un faux positif, sur les cinq datasets.")
with checked("M08.26"):
    select_features(X_26, y_26)                       # ⏳ while the function is not written
    wbm.check("M08.26", bool(ok_26), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu poser $H_0$ et $H_1$, et dire ce qu'une p-valeur mesure et ce qu'elle ne mesure pas ?
2. Sais-tu calculer la p-valeur exacte d'un test binomial et celle d'un petit test par permutation ?
3. Sais-tu calculer un risque de type I, une puissance, et le nombre d'observations nécessaire ?
4. Sais-tu calculer les intervalles de Wald, de Wilson et de Student apparié, et dire quand Wald échoue ?
5. Sais-tu corriger des comparaisons multiples par Bonferroni, et comparer deux modèles par l'AIC, le BIC et un facteur de Bayes ?

**La suite** : le chapitre 8 de workbookIA (entraînement et test : jeu de test, validation croisée, comparer deux modèles), puis M09a, les matrices et les systèmes linéaires.